# Causal Scrubbing & Interventions

Go beyond simple ablation with **causal scrubbing** — replacing activations
with values from an independent reference run. This is more rigorous than
zero ablation because it uses activations from a natural distribution.

This notebook covers the `irtk.causal_scrubbing` module.

## Why This Matters

Causal scrubbing is a rigorous method for testing whether a hypothesized circuit fully explains a model's behavior. By replacing non-circuit activations with those from unrelated inputs, it tests whether the circuit is sufficient — a stronger claim than correlation.

**Key references:**
- [Wang et al. (2023) "Interpretability in the Wild: IOI"](https://arxiv.org/abs/2211.00593) — Detailed circuit analysis of indirect object identification
- [Conmy et al. (2023) "Towards Automated Circuit Discovery"](https://arxiv.org/abs/2304.14997) — Automated methods for circuit finding via edge pruning

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import causal_scrubbing

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## 1. Basic Causal Scrubbing

Replace activations at selected hooks with values from an independent
reference input. If the metric is preserved, those hooks are NOT
part of the relevant circuit.

In [ ]:
clean_prompt = "The Eiffel Tower is located in"
ref_prompt = "My favorite color happens to be"

clean_tokens = model.to_tokens(clean_prompt)
ref_tokens = model.to_tokens(ref_prompt)
token_strs = [tokenizer.decode([int(t)]) for t in clean_tokens]

paris_id = tokenizer.encode(" Paris")[0]
def metric(logits):
    return float(logits[-1, paris_id])

# Scrub attention output at layers 0-2
hooks_early = [f"blocks.{i}.hook_attn_out" for i in range(3)]
result = causal_scrubbing.causal_scrub(model, clean_tokens, ref_tokens, hooks_early, metric)

print(f"Clean metric:     {result['clean_metric']:.4f}")
print(f"Scrubbed metric:  {result['scrubbed_metric']:.4f}")
print(f"Metric change:    {result['metric_change']:.4f}")
print(f"Relative change:  {result['relative_change']:.4f}")

In [ ]:
# Compare scrubbing early vs late layers
effects = []
for start in range(0, model.cfg.n_layers, 2):
    hooks = [f"blocks.{i}.hook_attn_out" for i in range(start, min(start+2, model.cfg.n_layers))]
    r = causal_scrubbing.causal_scrub(model, clean_tokens, ref_tokens, hooks, metric)
    effects.append((f"L{start}-{start+1}", r['relative_change']))

fig, ax = plt.subplots(figsize=(10, 4))
labels, vals = zip(*effects)
ax.bar(range(len(vals)), vals, color='steelblue')
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_ylabel('Relative metric change')
ax.set_title('Impact of Scrubbing Layer Pairs')
plt.tight_layout()
plt.show()

## 2. Interchange Intervention

Swap activations from a source input into the base run at a specific
hook point. Optionally restrict to specific positions.

In [ ]:
base_prompt = "The Eiffel Tower is located in"
source_prompt = "The Colosseum is located in"

base_tokens = model.to_tokens(base_prompt)
source_tokens = model.to_tokens(source_prompt)

# Full interchange at each layer
for layer in [0, 3, 6, 9, 11]:
    hook = f"blocks.{layer}.hook_attn_out"
    r = causal_scrubbing.interchange_intervention(
        model, base_tokens, source_tokens, hook, metric
    )
    print(f"Layer {layer:2d}: base={r['base_metric']:.3f}  "
          f"intervened={r['intervened_metric']:.3f}  "
          f"change={r['metric_change']:+.3f}")

In [ ]:
# Position-specific intervention at layer 9
token_strs_base = [tokenizer.decode([int(t)]) for t in base_tokens]
print(f"Tokens: {token_strs_base}")

for pos in range(len(base_tokens)):
    r = causal_scrubbing.interchange_intervention(
        model, base_tokens, source_tokens,
        "blocks.9.hook_attn_out", metric, positions=[pos]
    )
    print(f"  Swap pos {pos} ({token_strs_base[pos]:>10s}): change={r['metric_change']:+.4f}")

## 3. Path Patching Matrix

Compute the patching effect for all (sender, receiver) layer pairs.
This maps information flow between layers.

In [ ]:
corrupted_tokens = model.to_tokens("My favorite color happens to be")

result = causal_scrubbing.path_patching_matrix(
    model, clean_tokens, corrupted_tokens, metric
)

print(f"Clean metric:     {result['clean_metric']:.4f}")
print(f"Corrupted metric: {result['corrupted_metric']:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Patching matrix
im = axes[0].imshow(result['matrix'], cmap='RdBu_r', aspect='auto')
axes[0].set_xlabel('Receiver layer')
axes[0].set_ylabel('Sender layer')
axes[0].set_title('Path Patching Matrix')
plt.colorbar(im, ax=axes[0], label='Metric change')

# Layer effects
axes[1].bar(range(model.cfg.n_layers), result['layer_effects'], color='coral')
axes[1].set_xlabel('Layer')
axes[1].set_ylabel('Effect of patching layer alone')
axes[1].set_title('Single-Layer Patching Effects')

plt.tight_layout()
plt.show()

## 4. Corrupt and Restore

Corrupt a hook point, then restore clean values at each downstream
layer. This reveals where the model has backup pathways.

In [ ]:
result = causal_scrubbing.corrupt_and_restore(
    model, clean_tokens, "blocks.0.hook_attn_out", metric
)

print(f"Clean metric:     {result['clean_metric']:.4f}")
print(f"Corrupted metric: {result['corrupted_metric']:.4f}")

fig, ax = plt.subplots(figsize=(12, 4))
x = range(model.cfg.n_layers)
ax.bar(x, result['recovery_at_layer'], color='steelblue')
ax.axhline(result['clean_metric'] - result['corrupted_metric'],
           color='red', linestyle='--', label='Full recovery target')
ax.set_xlabel('Restore layer')
ax.set_ylabel('Recovery (restored - corrupted)')
ax.set_title('Recovery After Corrupting Layer 0 Attention')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Multi-Hook Scrub

Scrub hooks one at a time to rank their importance for the circuit.

In [ ]:
all_attn_hooks = [f"blocks.{i}.hook_attn_out" for i in range(model.cfg.n_layers)]
result = causal_scrubbing.multi_hook_scrub(
    model, clean_tokens, ref_tokens, all_attn_hooks, metric
)

print(f"Most important hook: {result['most_important']}")
print(f"\nPer-hook effects:")

effects = sorted(result['per_hook_effects'].items(), key=lambda x: abs(x[1]), reverse=True)
for hook, effect in effects:
    layer = hook.split('.')[1]
    bar = '|' * int(abs(effect) * 20)
    sign = '+' if effect > 0 else '-'
    print(f"  L{layer:>2s}: {effect:+.4f} {bar}")

## Summary

| Function | Purpose |
|----------|--------|
| `causal_scrub()` | Replace activations with independent reference values |
| `interchange_intervention()` | Swap activations between runs at a hook |
| `path_patching_matrix()` | All (sender, receiver) layer pair effects |
| `corrupt_and_restore()` | Corrupt one hook, restore at each downstream layer |
| `multi_hook_scrub()` | Scrub hooks one at a time to rank importance |